# Influencia de la carga competitiva acumulada en el rendimiento de tenistas profesionales

## Fase III. Construcción de las variables de carga competitiva

En esta fase construyo variables que representan el volumen competitivo acumulado por cada jugador antes del inicio de un torneo.

El dataset de partida se encuentra organizado a nivel jugador–partido. Cada encuentro aparece mediante dos observaciones, una desde la perspectiva de cada participante.

Como la fecha disponible corresponde al torneo y no al día exacto de cada partido, adaptaré temporalmente la información mediante dos estructuras intermedias:

$
\text{jugador–partido}
\longrightarrow
\text{jugador–torneo}
\longrightarrow
\text{jugador–fecha}.$


La unidad final del estudio continuará siendo jugador–partido. Las tablas intermedias se utilizarán exclusivamente para construir las variables históricas sin imponer un orden temporal que no pueda justificarse con los datos.

In [1]:
# Importa pandas para cargar, manipular y transformar datasets en formato tabla.
import pandas as pd

In [2]:
# Defino la ruta del dataset limpio.
file_path = "df_jugador_limpio.csv"

# Cargo el dataset que utilizaré como punto de partida.
data = pd.read_csv(
    file_path,
    low_memory=False
)

# Convierto la fecha de referencia del torneo.
data["fecha_torneo"] = pd.to_datetime(
    data["fecha_torneo"],
    errors="raise"
)

# Guardo dimensiones iniciales para validar que las cargas
# no alteran el número de observaciones ni de partidos.
filas_iniciales = len(data)
columnas_iniciales = data.shape[1]
partidos_iniciales = data["_id_partido"].nunique()

print("Dimensiones:", data.shape)
print("Número de partidos:", partidos_iniciales)

# Comprobaciones esperadas tras 03_validacion_calidad_datos.ipynb.
assert filas_iniciales > 0
assert columnas_iniciales > 0
assert partidos_iniciales > 0

columnas_minimas = [
    "_id_partido",
    "jugador_id",
    "identificador_torneo",
    "fecha_torneo",
    "match_year",
    "total_juegos",
    "total_sets",
    "porcentaje_juegos_ganados",
    "es_muestra_estudio",
    "es_buffer_carga",
    "es_desarrollo_modelo",
    "es_validacion_temporal"
]

assert set(columnas_minimas).issubset(data.columns), (
    "Faltan columnas mínimas necesarias para construir las cargas."
)

assert (
    data.groupby("_id_partido")
    .size()
    .eq(2)
    .all()
), (
    "Algún partido no conserva exactamente dos perspectivas jugador-partido."
)

# Compruebo que se conservan las columnas temporales nuevas.
columnas_temporales = [
    "es_muestra_estudio",
    "es_buffer_carga",
    "es_desarrollo_modelo",
    "es_validacion_temporal"
]

assert set(columnas_temporales).issubset(data.columns)

assert (
    data["es_buffer_carga"]
    + data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(1).all()

assert (
    data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(data["es_muestra_estudio"]).all()

assert 2020 not in data["match_year"].unique()

print("Dataset limpio cargado correctamente.")

Dimensiones: (94030, 79)
Número de partidos: 47015
Dataset limpio cargado correctamente.


## 3. Definición operacional de la carga competitiva acumulada

En este trabajo defino la carga competitiva como la **carga competitiva externa observada**, entendida como el volumen de competición oficial disputado por cada jugador antes del comienzo del torneo actual.

Esta definición no representa directamente la fatiga ni la carga fisiológica real del tenista. La base de datos no contiene información sobre entrenamientos, desplazamientos, recuperación, percepción subjetiva del esfuerzo, estado médico o intensidad física. Por tanto, las variables construidas deben interpretarse como indicadores de **exposición competitiva acumulada**.

Utilizaré los **juegos disputados** como medida principal de carga, porque permiten diferenciar partidos de distinta extensión. Los sets y los partidos acumulados se utilizarán como medidas complementarias, mientras que el número de torneos representará la frecuencia de participación competitiva.

La carga se calculará antes del inicio del torneo actual. Para un jugador (i), únicamente consideraré actividad registrada en fechas estrictamente anteriores:

$$
T_k < T_t
$$

donde ($T_k$) representa la fecha de una participación anterior y ($T_t$) la fecha del torneo actual.

Esta condición evita que el torneo actual contribuya a su propia carga acumulada.

La definición principal será la carga registrada durante los 365 días anteriores:

$$
C_{i,t}^{365}(z)
=
\sum_k z_{i,k}
\mathbb{1}
\left(
T_t - 365 \leq T_k < T_t
\right)
$$

donde ($z_{i,k}$) representa los juegos, sets, partidos o torneos disputados por el jugador ($i$) en una fecha anterior.

También construiré dos definiciones adicionales:

* carga acumulada desde el comienzo de la temporada competitiva;
* carga acumulada durante los 180 días anteriores, utilizada como análisis de robustez.

No calcularé carga dentro del propio torneo ni días exactos de descanso, porque la base de datos no contiene la fecha individual de cada partido.


In [3]:
# ============================================================
# CONSTRUCCIÓN DE LA TABLA JUGADOR–TORNEO
# ============================================================

# Agrupo todos los partidos disputados por un jugador
# dentro de un mismo torneo.
df_jugador_torneo = (
    data
    .groupby(
        [
            "jugador_id",
            "identificador_torneo"
        ],
        as_index=False,
        sort=False
    )
    .agg(
        jugador_nombre=(
            "jugador_nombre",
            "first"
        ),
        nombre_torneo=(
            "nombre_torneo",
            "first"
        ),
        fecha_torneo=(
            "fecha_torneo",
            "first"
        ),
        match_year=(
            "match_year",
            "first"
        ),
        es_muestra_estudio=(
            "es_muestra_estudio",
            "first"
        ),
        es_buffer_carga=(
            "es_buffer_carga",
            "first"
        ),
        es_desarrollo_modelo=(
            "es_desarrollo_modelo",
            "first"
        ),
        es_validacion_temporal=(
            "es_validacion_temporal",
            "first"
        ),
        superficie=(
            "superficie",
            "first"
        ),
        nivel_torneo=(
            "nivel_torneo",
            "first"
        ),
        partidos_torneo=(
            "_id_partido",
            "nunique"
        ),
        sets_torneo=(
            "total_sets",
            "sum"
        ),
        juegos_torneo=(
            "total_juegos",
            "sum"
        )
    )
)


# Cada fila representa una participación en un torneo.
df_jugador_torneo["torneos_torneo"] = 1

# Compruebo que las variables temporales son constantes
# dentro de cada jugador y torneo.
control_temporal_torneo = (
    data
    .groupby(
        [
            "jugador_id",
            "identificador_torneo"
        ]
    )[columnas_temporales + ["match_year"]]
    .nunique(dropna=False)
)

assert control_temporal_torneo.le(1).all().all(), (
    "Hay algún jugador–torneo con asignación temporal no constante."
)

# Compruebo que cada combinación jugador–torneo
# aparece una única vez en la estructura agregada.
assert not df_jugador_torneo.duplicated(
    subset=[
        "jugador_id",
        "identificador_torneo"
    ]
).any(), (
    "Existen duplicados en la clave jugador–torneo."
)

# Ordeno cronológicamente el historial de cada jugador.
df_jugador_torneo = (
    df_jugador_torneo
    .sort_values(
        [
            "jugador_id",
            "fecha_torneo",
            "identificador_torneo"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Dimensiones de la tabla jugador–torneo:",
    df_jugador_torneo.shape
)

display(df_jugador_torneo.head())

Dimensiones de la tabla jugador–torneo: (50051, 16)


,jugador_id,identificador_torneo,jugador_nombre,nombre_torneo,fecha_torneo,match_year,es_muestra_estudio,es_buffer_carga,es_desarrollo_modelo,es_validacion_temporal,superficie,nivel_torneo,partidos_torneo,sets_torneo,juegos_torneo,torneos_torneo
0,100644,2013-414,Alexander Zverev,Hamburg,2013-07-15,2013,1,0,1,0,Clay,ATP,1,2,17,1
1,100644,2014-308,Alexander Zverev,Munich,2014-04-28,2014,1,0,1,0,Clay,ATP,1,2,15,1
2,100644,2014-321,Alexander Zverev,Stuttgart,2014-07-07,2014,1,0,1,0,Clay,ATP,1,2,26,1
3,100644,2014-414,Alexander Zverev,Hamburg,2014-07-14,2014,1,0,1,0,Clay,ATP,5,11,101,1
4,100644,2014-439,Alexander Zverev,Umag,2014-07-21,2014,1,0,1,0,Clay,ATP,1,2,20,1


## Construcción de la tabla jugador–fecha

La tabla `df_jugador_torneo` contiene una fila por cada participación de un jugador en un torneo. Sin embargo, se ha identificado un caso en el que un mismo jugador aparece en dos torneos diferentes con la misma `fecha_torneo`.

Como la base de datos no permite establecer cuál de esos torneos ocurrió primero, no debo imponer un orden artificial entre ambas participaciones. Si calculase directamente la carga sobre `df_jugador_torneo`, uno de los torneos podría incorporarse erróneamente al historial previo del otro.

Por este motivo, construyo una tabla adicional denominada `df_jugador_fecha`, con una sola fila por jugador y fecha. En ella agrego conjuntamente:

* los partidos disputados;
* los sets disputados;
* los juegos disputados;
* el número de torneos registrados en esa fecha.

Esta tabla será la utilizada para calcular las cargas acumuladas. Los torneos que comparten fecha recibirán así la misma carga previa y ninguno formará parte del historial del otro.


In [4]:

# ============================================================
# CONSTRUCCIÓN DE LA TABLA JUGADOR–FECHA
# ============================================================

# Agrupo toda la actividad de un jugador registrada
# en una misma fecha.
df_jugador_fecha = (
    df_jugador_torneo
    .groupby(
        [
            "jugador_id",
            "fecha_torneo"
        ],
        as_index=False,
        sort=False
    )
    .agg(
        jugador_nombre=(
            "jugador_nombre",
            "first"
        ),
        match_year=(
            "match_year",
            "first"
        ),
        es_muestra_estudio=(
            "es_muestra_estudio",
            "first"
        ),
        es_buffer_carga=(
            "es_buffer_carga",
            "first"
        ),
        es_desarrollo_modelo=(
            "es_desarrollo_modelo",
            "first"
        ),
        es_validacion_temporal=(
            "es_validacion_temporal",
            "first"
        ),
        partidos_fecha=(
            "partidos_torneo",
            "sum"
        ),
        sets_fecha=(
            "sets_torneo",
            "sum"
        ),
        juegos_fecha=(
            "juegos_torneo",
            "sum"
        ),
        torneos_fecha=(
            "torneos_torneo",
            "sum"
        )
    )
)
# Compruebo que no hay contradicciones temporales
# al agrupar varios torneos de un mismo jugador en una misma fecha.
control_temporal_fecha = (
    df_jugador_torneo
    .groupby(
        [
            "jugador_id",
            "fecha_torneo"
        ]
    )[columnas_temporales + ["match_year"]]
    .nunique(dropna=False)
)

assert control_temporal_fecha.le(1).all().all(), (
    "Hay algún jugador–fecha con asignación temporal no constante."
)

# Compruebo que cada combinación jugador–fecha
# aparece una única vez en la estructura agregada.
assert not df_jugador_fecha.duplicated(
    subset=[
        "jugador_id",
        "fecha_torneo"
    ]
).any(), (
    "Existen duplicados en la clave jugador–fecha."
)

# Ordeno cronológicamente el historial de cada jugador.
df_jugador_fecha = (
    df_jugador_fecha
    .sort_values(
        [
            "jugador_id",
            "fecha_torneo"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Dimensiones de la tabla jugador–fecha:",
    df_jugador_fecha.shape
)

display(df_jugador_fecha.head())



Dimensiones de la tabla jugador–fecha: (50050, 12)


,jugador_id,fecha_torneo,jugador_nombre,match_year,es_muestra_estudio,es_buffer_carga,es_desarrollo_modelo,es_validacion_temporal,partidos_fecha,sets_fecha,juegos_fecha,torneos_fecha
0,100644,2013-07-15,Alexander Zverev,2013,1,0,1,0,1,2,17,1
1,100644,2014-04-28,Alexander Zverev,2014,1,0,1,0,1,2,15,1
2,100644,2014-07-07,Alexander Zverev,2014,1,0,1,0,1,2,26,1
3,100644,2014-07-14,Alexander Zverev,2014,1,0,1,0,5,11,101,1
4,100644,2014-07-21,Alexander Zverev,2014,1,0,1,0,1,2,20,1


In [5]:
# ============================================================
# COMPROBACIÓN DE LA SEPARACIÓN ENTRE BLOQUES TEMPORALES
# ============================================================

# Última fecha observable del primer bloque operativo.
fecha_fin_bloque_2008_2019 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2008, 2019),
        "fecha_torneo"
    ]
    .max()
)

# Primera fecha observable del segundo bloque operativo.
fecha_inicio_bloque_2021_2024 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2021, 2024),
        "fecha_torneo"
    ]
    .min()
)

separacion_bloques_dias = (
    fecha_inicio_bloque_2021_2024
    - fecha_fin_bloque_2008_2019
).days

print(
    "Última fecha del bloque 2008–2019:",
    fecha_fin_bloque_2008_2019
)

print(
    "Primera fecha del bloque 2021–2024:",
    fecha_inicio_bloque_2021_2024
)

print(
    "Separación entre bloques:",
    separacion_bloques_dias,
    "días"
)

# La separación debe superar la ventana temporal más larga
# utilizada en este notebook.
assert separacion_bloques_dias > 365, (
    "La separación entre bloques no supera los 365 días. "
    "Las ventanas móviles deberían calcularse explícitamente "
    "por bloque temporal."
)

print(
    "La separación entre bloques impide que actividad del "
    "periodo 2008–2019 entre en las ventanas de 2021–2024."
)

Última fecha del bloque 2008–2019: 2019-11-24 00:00:00
Primera fecha del bloque 2021–2024: 2021-01-04 00:00:00
Separación entre bloques: 407 días
La separación entre bloques impide que actividad del periodo 2008–2019 entre en las ventanas de 2021–2024.


## Construcción de la carga acumulada en los 365 días anteriores

A partir de `df_jugador_fecha` calculo la carga competitiva acumulada por cada jugador durante los 365 días anteriores a cada fecha de torneo.

Para cada observación sumaré:

* los juegos disputados;
* los sets disputados;
* los partidos disputados;
* los torneos disputados.

La ventana temporal se define como:

$$
T_t - 365 \leq T_k < T_t
$$

Por tanto, incluyo toda la actividad registrada durante los 365 días anteriores, pero excluyo siempre la fecha actual. De esta forma, el torneo que voy a predecir no interviene en la construcción de su propia carga.

Las variables obtenidas serán:

* `juegos_365d`;
* `sets_365d`;
* `partidos_365d`;
* `torneos_365d`.

Estas variables se calcularán primero a nivel jugador–fecha y posteriormente se trasladarán a la tabla jugador–torneo y al dataset jugador–partido.


In [6]:

# ============================================================
# CARGA ACUMULADA EN LOS 365 DÍAS ANTERIORES
# ============================================================

# Variables de volumen que acumularé.
columnas_volumen = [
    "juegos_fecha",
    "sets_fecha",
    "partidos_fecha",
    "torneos_fecha"
]


# Calculo, para cada jugador, la suma de la actividad
# registrada en los 365 días anteriores.
#
# closed="left" establece la ventana:
# [fecha actual - 365 días, fecha actual)
#
# Por tanto, incluye el límite de hace 365 días
# y excluye completamente la fecha actual.
cargas_365d = (
    df_jugador_fecha
    .set_index("fecha_torneo")
    .groupby("jugador_id")[columnas_volumen]
    .rolling(
        window="365D",
        closed="left"
    )
    .sum()
    .reset_index()
)


# Renombro las variables acumuladas.
cargas_365d = cargas_365d.rename(
    columns={
        "juegos_fecha": "juegos_365d",
        "sets_fecha": "sets_365d",
        "partidos_fecha": "partidos_365d",
        "torneos_fecha": "torneos_365d"
    }
)


# Incorporo las cargas a la tabla jugador–fecha.
df_jugador_fecha = df_jugador_fecha.merge(
    cargas_365d,
    on=[
        "jugador_id",
        "fecha_torneo"
    ],
    how="left",
    validate="one_to_one"
)


# Cuando no existe actividad anterior en la ventana,
# rolling devuelve NaN. Lo transformo en cero.
columnas_carga_365d = [
    "juegos_365d",
    "sets_365d",
    "partidos_365d",
    "torneos_365d"
]

df_jugador_fecha[columnas_carga_365d] = (
    df_jugador_fecha[columnas_carga_365d]
    .fillna(0)
    .astype("int64")
)


# Identifico el comienzo operativo de cada bloque temporal.
# 2008 y 2021 se utilizan como años buffer para construir historial.
fecha_inicio_bloque_2008_2019 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2008, 2019),
        "fecha_torneo"
    ]
    .min()
)

fecha_inicio_bloque_2021_2024 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2021, 2024),
        "fecha_torneo"
    ]
    .min()
)

# Distingo las observaciones de la muestra de estudio
# con 365 días completos de historial potencialmente observable.
df_jugador_fecha["historial_365d_completo"] = (
    (
        df_jugador_fecha["match_year"].between(2009, 2019)
        &
        (
            df_jugador_fecha["fecha_torneo"]
            >= fecha_inicio_bloque_2008_2019 + pd.Timedelta(days=365)
        )
    )
    |
    (
        df_jugador_fecha["match_year"].between(2022, 2024)
        &
        (
            df_jugador_fecha["fecha_torneo"]
            >= fecha_inicio_bloque_2021_2024 + pd.Timedelta(days=365)
        )
    )
)

# Los años buffer se usan para calcular cargas,
# pero no forman parte de la muestra de estudio.
df_jugador_fecha.loc[
    df_jugador_fecha["es_buffer_carga"].eq(1),
    "historial_365d_completo"
] = False

# Resumo la cobertura únicamente dentro de la muestra de estudio.
mascara_muestra_estudio = (
    df_jugador_fecha["es_muestra_estudio"].eq(1)
)

n_muestra_estudio = int(mascara_muestra_estudio.sum())

n_historial_365d_completo = int(
    df_jugador_fecha.loc[
        mascara_muestra_estudio,
        "historial_365d_completo"
    ].sum()
)

n_historial_365d_incompleto = (
    n_muestra_estudio - n_historial_365d_completo
)

print(
    "Observaciones jugador-fecha de la muestra de estudio:",
    n_muestra_estudio
)

print(
    "Con historial completo de 365 días:",
    n_historial_365d_completo
)

print(
    "Con historial incompleto de 365 días:",
    n_historial_365d_incompleto
)


display(
    df_jugador_fecha[
        [
            "jugador_id",
            "jugador_nombre",
            "fecha_torneo",
            "juegos_fecha",
            "juegos_365d",
            "sets_365d",
            "partidos_365d",
            "torneos_365d",
            "historial_365d_completo"
        ]
    ].head(10)
)



Observaciones jugador-fecha de la muestra de estudio: 43863
Con historial completo de 365 días: 43770
Con historial incompleto de 365 días: 93


,jugador_id,jugador_nombre,fecha_torneo,juegos_fecha,juegos_365d,sets_365d,partidos_365d,torneos_365d,historial_365d_completo
0,100644,Alexander Zverev,2013-07-15,17,0,0,0,0,True
1,100644,Alexander Zverev,2014-04-28,15,17,2,1,1,True
2,100644,Alexander Zverev,2014-07-07,26,32,4,2,2,True
3,100644,Alexander Zverev,2014-07-14,101,58,6,3,3,True
4,100644,Alexander Zverev,2014-07-21,20,142,15,7,3,True
5,100644,Alexander Zverev,2014-07-27,13,162,17,8,4,True
6,100644,Alexander Zverev,2014-10-20,26,175,19,9,5,True
7,100644,Alexander Zverev,2015-02-09,17,201,22,10,6,True
8,100644,Alexander Zverev,2015-02-16,20,218,24,11,7,True
9,100644,Alexander Zverev,2015-02-23,20,238,26,12,8,True


## Incorporación de la carga de 365 días a la tabla jugador–torneo

Las cargas acumuladas se han calculado a nivel jugador–fecha, porque esta estructura evita ordenar artificialmente torneos que comparten la misma fecha.

Ahora traslado estas variables a `df_jugador_torneo` utilizando como claves:

* `jugador_id`;
* `fecha_torneo`.

De esta forma, todos los torneos disputados por un jugador en una misma fecha reciben exactamente la misma carga previa.

Esta unión no modifica los valores de carga ni añade información del torneo actual. Únicamente devuelve las variables históricas a una estructura en la que cada fila representa la participación de un jugador en un torneo.


In [7]:

# ============================================================
# INCORPORACIÓN DE LA CARGA DE 365 DÍAS A df_jugador_torneo
# ============================================================

# Selecciono únicamente las claves y las variables
# históricas calculadas a nivel jugador–fecha.
cargas_365d_jugador_fecha = df_jugador_fecha[
    [
        "jugador_id",
        "fecha_torneo",
        "juegos_365d",
        "sets_365d",
        "partidos_365d",
        "torneos_365d",
        "historial_365d_completo"
    ]
].copy()


# Incorporo la misma carga previa a todos los torneos
# disputados por un jugador en una misma fecha.
df_jugador_torneo = df_jugador_torneo.merge(
    cargas_365d_jugador_fecha,
    on=[
        "jugador_id",
        "fecha_torneo"
    ],
    how="left",
    validate="many_to_one"
)


print(
    "Dimensiones de df_jugador_torneo:",
    df_jugador_torneo.shape
)


display(
    df_jugador_torneo[
        [
            "jugador_id",
            "jugador_nombre",
            "nombre_torneo",
            "fecha_torneo",
            "juegos_torneo",
            "juegos_365d",
            "sets_365d",
            "partidos_365d",
            "torneos_365d",
            "historial_365d_completo"
        ]
    ].head(10)
)



Dimensiones de df_jugador_torneo: (50051, 21)


,jugador_id,jugador_nombre,nombre_torneo,fecha_torneo,juegos_torneo,juegos_365d,sets_365d,partidos_365d,torneos_365d,historial_365d_completo
0,100644,Alexander Zverev,Hamburg,2013-07-15,17,0,0,0,0,True
1,100644,Alexander Zverev,Munich,2014-04-28,15,17,2,1,1,True
2,100644,Alexander Zverev,Stuttgart,2014-07-07,26,32,4,2,2,True
3,100644,Alexander Zverev,Hamburg,2014-07-14,101,58,6,3,3,True
4,100644,Alexander Zverev,Umag,2014-07-21,20,142,15,7,3,True
5,100644,Alexander Zverev,Kitzbuhel,2014-07-27,13,162,17,8,4,True
6,100644,Alexander Zverev,Basel,2014-10-20,26,175,19,9,5,True
7,100644,Alexander Zverev,Rotterdam,2015-02-09,17,201,22,10,6,True
8,100644,Alexander Zverev,Marseille,2015-02-16,20,218,24,11,7,True
9,100644,Alexander Zverev,Dubai,2015-02-23,20,238,26,12,8,True


## Incorporación de la carga de 365 días al dataset jugador–partido

La unidad final del análisis es jugador–partido. Por ello, traslado ahora las variables de carga desde `df_jugador_torneo` hasta el DataFrame original `data`.

La unión se realiza mediante:

* `jugador_id`;
* `identificador_torneo`.

Cada partido disputado por un jugador dentro de un mismo torneo recibe la misma carga competitiva previa. Esto es coherente con la definición adoptada, ya que la carga se mide antes del comienzo del torneo y no se actualiza entre rondas.

Como resultado, las variables `juegos_365d`, `sets_365d`, `partidos_365d` y `torneos_365d` podrán utilizarse posteriormente como predictores del rendimiento observado en cada partido.

El indicador `historial_365d_completo` permitirá distinguir las observaciones cuya ventana anual se encuentra completamente cubierta por el periodo disponible.


In [8]:

# ============================================================
# INCORPORACIÓN DE LA CARGA DE 365 DÍAS A data
# ============================================================

# Selecciono una única fila por jugador y torneo
# con las variables de carga previamente calculadas.
cargas_365d_jugador_torneo = df_jugador_torneo[
    [
        "jugador_id",
        "identificador_torneo",
        "juegos_365d",
        "sets_365d",
        "partidos_365d",
        "torneos_365d",
        "historial_365d_completo"
    ]
].copy()


# Incorporo la carga pretorneo a cada observación
# jugador–partido del dataset principal.
data = data.merge(
    cargas_365d_jugador_torneo,
    on=[
        "jugador_id",
        "identificador_torneo"
    ],
    how="left",
    validate="many_to_one"
)


print("Dimensiones del dataset actualizado:", data.shape)


display(
    data[
        [
            "_id_partido",
            "jugador_id",
            "jugador_nombre",
            "rival_id",
            "identificador_torneo",
            "nombre_torneo",
            "fecha_torneo",
            "juegos_365d",
            "sets_365d",
            "partidos_365d",
            "torneos_365d",
            "historial_365d_completo",
            "porcentaje_juegos_ganados"
        ]
    ].head(10)
)



Dimensiones del dataset actualizado: (94030, 84)


,_id_partido,jugador_id,jugador_nombre,rival_id,identificador_torneo,nombre_torneo,fecha_torneo,juegos_365d,sets_365d,partidos_365d,torneos_365d,historial_365d_completo,porcentaje_juegos_ganados
0,2008-339__1,104534,Dudi Sela,103720,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.250000
1,2008-339__1,103720,Lleyton Hewitt,104534,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.750000
2,2008-339__2,104268,Alejandro Falla,104076,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.409091
3,2008-339__2,104076,Jose Acasuso,104268,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.590909
4,2008-339__3,104979,Andrey Golubev,105208,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.419355
5,2008-339__3,105208,Ernests Gulbis,104979,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.580645
6,2008-339__4,104542,Jo-Wilfried Tsonga,103812,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.548387
7,2008-339__4,103812,Victor Hanescu,104542,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.451613
8,2008-339__5,103657,Ivo Klec,103813,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.333333
9,2008-339__5,103813,Jarkko Nieminen,103657,2008-339,Adelaide,2007-12-31,0,0,0,0,False,0.666667


## Comprobación de las variables de carga de 365 días

Antes de construir las siguientes definiciones de carga, compruebo que la incorporación de las variables anuales se ha realizado correctamente.

Verifico que:

* no existan valores ausentes en las cargas calculadas;
* las cargas no presenten valores negativos;
* todos los partidos de un jugador dentro del mismo torneo reciban la misma carga previa;
* el número de observaciones del dataset principal se mantenga constante.

Los valores iguales a cero son posibles y no implican necesariamente un error. Pueden corresponder al primer torneo observado de un jugador o a un periodo de 365 días sin actividad registrada. Su interpretación dependerá del indicador `historial_365d_completo`.


In [9]:

# ============================================================
# COMPROBACIÓN DE LAS VARIABLES DE CARGA DE 365 DÍAS
# ============================================================

columnas_carga_365d = [
    "juegos_365d",
    "sets_365d",
    "partidos_365d",
    "torneos_365d"
]


# Compruebo que la unión no ha modificado
# el número de observaciones del dataset.
assert len(data) == filas_iniciales

assert data["_id_partido"].nunique() == partidos_iniciales

assert (
    data["es_buffer_carga"]
    + data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(1).all()

assert (
    data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(data["es_muestra_estudio"]).all()


# Compruebo que las variables construidas
# no contienen valores ausentes ni negativos.
assert data[columnas_carga_365d].notna().all().all()

assert data[columnas_carga_365d].ge(0).all().all()

assert data["historial_365d_completo"].notna().all()


# Compruebo que todos los partidos de un jugador
# dentro del mismo torneo reciben la misma carga previa.
columnas_comprobacion = (
    columnas_carga_365d
    + ["historial_365d_completo"]
)

variacion_dentro_torneo = (
    data
    .groupby(
        [
            "jugador_id",
            "identificador_torneo"
        ]
    )[columnas_comprobacion]
    .nunique()
)

assert variacion_dentro_torneo.le(1).all().all(), (
    "Existen partidos del mismo jugador y torneo "
    "con cargas previas diferentes."
)


print("Comprobaciones superadas.")

print(
    "Observaciones con historial completo:",
    data["historial_365d_completo"].sum()
)

print(
    "Observaciones con historial incompleto:",
    (~data["historial_365d_completo"]).sum()
)


display(
    data.loc[
        data["historial_365d_completo"],
        columnas_carga_365d
    ].describe().T
)



Comprobaciones superadas.
Observaciones con historial completo: 82188
Observaciones con historial incompleto: 11842


,count,mean,std,min,25%,50%,75%,max
juegos_365d,82188.0,987.942303,592.718530,0.0,494.0,1044.0,1437.0,2556.0
sets_365d,82188.0,99.985314,60.049228,0.0,50.0,105.0,145.0,255.0
partidos_365d,82188.0,38.499136,23.063061,0.0,19.0,41.0,56.0,97.0
torneos_365d,82188.0,17.778654,9.004505,0.0,11.0,20.0,25.0,35.0


## Construcción de la carga acumulada durante la temporada

Además de la ventana móvil de 365 días, calculo la actividad competitiva acumulada por cada jugador desde el comienzo de la temporada hasta la fecha actual.

Para una participación perteneciente a la temporada ($s$), la carga se define como:

$$
C_{i,t}^{\text{temporada}}(z)
=
\sum_k z_{i,k}
\mathbb{1}
\left(
\text{temporada}_k = \text{temporada}_t
\right)
\mathbb{1}
\left(
T_k < T_t
\right)
$$

Únicamente incluyo actividad de la misma temporada y de fechas estrictamente anteriores. Por tanto, la participación actual no forma parte de su propia carga.

Para identificar la temporada utilizo `match_year` y no el año natural de `fecha_torneo`. Esta decisión permite mantener correctamente dentro de una temporada los torneos que comienzan durante los últimos días del año anterior.

Construiré las siguientes variables:

* `juegos_temporada`;
* `sets_temporada`;
* `partidos_temporada`;
* `torneos_temporada`.

A diferencia de la ventana de 365 días, esta definición no requiere un periodo de calentamiento. La carga igual a cero en el primer torneo de una temporada es un resultado correcto, porque la acumulación se reinicia deliberadamente al comenzar cada temporada.


In [10]:

# ============================================================
# CARGA ACUMULADA DURANTE LA TEMPORADA
# ============================================================

# Ordeno las observaciones dentro de cada jugador y temporada.
df_jugador_fecha = (
    df_jugador_fecha
    .sort_values(
        [
            "jugador_id",
            "match_year",
            "fecha_torneo"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)


columnas_volumen_fecha = [
    "juegos_fecha",
    "sets_fecha",
    "partidos_fecha",
    "torneos_fecha"
]


# Calculo la suma acumulada dentro de cada temporada.
# Resto el volumen de la fecha actual para obtener
# únicamente la actividad registrada en fechas anteriores.
cargas_temporada = (
    df_jugador_fecha
    .groupby(
        [
            "jugador_id",
            "match_year"
        ],
        sort=False
    )[columnas_volumen_fecha]
    .cumsum()
    - df_jugador_fecha[columnas_volumen_fecha]
)


# Renombro las variables acumuladas.
cargas_temporada = cargas_temporada.rename(
    columns={
        "juegos_fecha": "juegos_temporada",
        "sets_fecha": "sets_temporada",
        "partidos_fecha": "partidos_temporada",
        "torneos_fecha": "torneos_temporada"
    }
)


# Incorporo las cargas de temporada a df_jugador_fecha.
columnas_carga_temporada = [
    "juegos_temporada",
    "sets_temporada",
    "partidos_temporada",
    "torneos_temporada"
]

df_jugador_fecha[columnas_carga_temporada] = (
    cargas_temporada[columnas_carga_temporada]
    .astype("int64")
)


# Compruebo que las cargas no sean negativas.
assert (
    df_jugador_fecha[
        columnas_carga_temporada
    ]
    .ge(0)
    .all()
    .all()
)


# El primer registro observado de cada jugador
# dentro de cada temporada debe tener carga igual a cero.
primera_fecha_temporada = (
    ~df_jugador_fecha.duplicated(
        subset=[
            "jugador_id",
            "match_year"
        ]
    )
)

assert (
    df_jugador_fecha.loc[
        primera_fecha_temporada,
        columnas_carga_temporada
    ]
    .eq(0)
    .all()
    .all()
)


print(
    "He calculado correctamente la carga "
    "acumulada durante la temporada."
)


display(
    df_jugador_fecha[
        [
            "jugador_id",
            "jugador_nombre",
            "match_year",
            "fecha_torneo",
            "juegos_fecha",
            "juegos_temporada",
            "sets_temporada",
            "partidos_temporada",
            "torneos_temporada"
        ]
    ].head(15)
)



He calculado correctamente la carga acumulada durante la temporada.


,jugador_id,jugador_nombre,match_year,fecha_torneo,juegos_fecha,juegos_temporada,sets_temporada,partidos_temporada,torneos_temporada
0,100644,Alexander Zverev,2013,2013-07-15,17,0,0,0,0
1,100644,Alexander Zverev,2014,2014-04-28,15,0,0,0,0
2,100644,Alexander Zverev,2014,2014-07-07,26,15,2,1,1
3,100644,Alexander Zverev,2014,2014-07-14,101,41,4,2,2
4,100644,Alexander Zverev,2014,2014-07-21,20,142,15,7,3
5,100644,Alexander Zverev,2014,2014-07-27,13,162,17,8,4
6,100644,Alexander Zverev,2014,2014-10-20,26,175,19,9,5
7,100644,Alexander Zverev,2015,2015-02-09,17,0,0,0,0
8,100644,Alexander Zverev,2015,2015-02-16,20,17,2,1,1
9,100644,Alexander Zverev,2015,2015-02-23,20,37,4,2,2


## Incorporación de la carga de temporada al dataset principal

Las variables de carga de temporada se han calculado a nivel jugador–fecha. Ahora las traslado primero a `df_jugador_torneo` y posteriormente a `data`.

La unión se realiza mediante `jugador_id` y `fecha_torneo`. De este modo, todos los partidos disputados por un jugador dentro del mismo torneo reciben la misma carga acumulada previa al comienzo de la competición.

Las variables incorporadas son:

* `juegos_temporada`;
* `sets_temporada`;
* `partidos_temporada`;
* `torneos_temporada`.

Estas variables solo contienen actividad registrada en fechas anteriores y perteneciente a la misma temporada competitiva.


In [11]:

# ============================================================
# INCORPORACIÓN DE LA CARGA DE TEMPORADA
# ============================================================

# Selecciono las cargas calculadas a nivel jugador–fecha.
cargas_temporada_jugador_fecha = df_jugador_fecha[
    [
        "jugador_id",
        "fecha_torneo",
        "juegos_temporada",
        "sets_temporada",
        "partidos_temporada",
        "torneos_temporada"
    ]
].copy()


# Evito duplicar columnas si vuelvo a ejecutar la celda.
df_jugador_torneo = df_jugador_torneo.drop(
    columns=columnas_carga_temporada,
    errors="ignore"
)


# Incorporo la carga de temporada a cada jugador–torneo.
df_jugador_torneo = df_jugador_torneo.merge(
    cargas_temporada_jugador_fecha,
    on=[
        "jugador_id",
        "fecha_torneo"
    ],
    how="left",
    validate="many_to_one"
)


# Selecciono una única fila de carga por jugador y torneo.
cargas_temporada_jugador_torneo = df_jugador_torneo[
    [
        "jugador_id",
        "identificador_torneo",
        "juegos_temporada",
        "sets_temporada",
        "partidos_temporada",
        "torneos_temporada"
    ]
].copy()


# Evito duplicar columnas si vuelvo a ejecutar la celda.
data = data.drop(
    columns=columnas_carga_temporada,
    errors="ignore"
)


# Incorporo las cargas al dataset jugador–partido.
data = data.merge(
    cargas_temporada_jugador_torneo,
    on=[
        "jugador_id",
        "identificador_torneo"
    ],
    how="left",
    validate="many_to_one"
)


print(
    "Dimensiones del dataset actualizado:",
    data.shape
)


display(
    data[
        [
            "_id_partido",
            "jugador_id",
            "jugador_nombre",
            "nombre_torneo",
            "fecha_torneo",
            "match_year",
            "juegos_temporada",
            "sets_temporada",
            "partidos_temporada",
            "torneos_temporada"
        ]
    ].head(10)
)



Dimensiones del dataset actualizado: (94030, 88)


,_id_partido,jugador_id,jugador_nombre,nombre_torneo,fecha_torneo,match_year,juegos_temporada,sets_temporada,partidos_temporada,torneos_temporada
0,2008-339__1,104534,Dudi Sela,Adelaide,2007-12-31,2008,0,0,0,0
1,2008-339__1,103720,Lleyton Hewitt,Adelaide,2007-12-31,2008,0,0,0,0
2,2008-339__2,104268,Alejandro Falla,Adelaide,2007-12-31,2008,0,0,0,0
3,2008-339__2,104076,Jose Acasuso,Adelaide,2007-12-31,2008,0,0,0,0
4,2008-339__3,104979,Andrey Golubev,Adelaide,2007-12-31,2008,0,0,0,0
5,2008-339__3,105208,Ernests Gulbis,Adelaide,2007-12-31,2008,0,0,0,0
6,2008-339__4,104542,Jo-Wilfried Tsonga,Adelaide,2007-12-31,2008,0,0,0,0
7,2008-339__4,103812,Victor Hanescu,Adelaide,2007-12-31,2008,0,0,0,0
8,2008-339__5,103657,Ivo Klec,Adelaide,2007-12-31,2008,0,0,0,0
9,2008-339__5,103813,Jarkko Nieminen,Adelaide,2007-12-31,2008,0,0,0,0


## Construcción de la carga acumulada en los 180 días anteriores

Como análisis de robustez, calculo también la actividad competitiva acumulada durante los 180 días anteriores a cada fecha de torneo.

La definición es:

$$
C_{i,t}^{180}(z)
=
\sum_k z_{i,k}
\mathbb{1}
\left(
T_t - 180 \leq T_k < T_t
\right)
$$

La lógica es la misma que en la ventana anual:

* incluyo únicamente actividad registrada en fechas anteriores;
* excluyo completamente la fecha actual;
* agrego juegos, sets, partidos y torneos;
* calculo las variables a nivel jugador–fecha.

Las nuevas variables serán:

* `juegos_180d`;
* `sets_180d`;
* `partidos_180d`;
* `torneos_180d`.

También crearé `historial_180d_completo`, que identificará las observaciones para las que se dispone de los 180 días completos de historial potencialmente observable.


In [12]:

# ============================================================
# CARGA ACUMULADA EN LOS 180 DÍAS ANTERIORES
# ============================================================

# Ordeno nuevamente por jugador y fecha antes
# de aplicar la ventana temporal móvil.
df_jugador_fecha = (
    df_jugador_fecha
    .sort_values(
        [
            "jugador_id",
            "fecha_torneo"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)


columnas_volumen_fecha = [
    "juegos_fecha",
    "sets_fecha",
    "partidos_fecha",
    "torneos_fecha"
]


# Calculo la actividad registrada en los 180 días anteriores.
# closed="left" excluye la fecha actual de la ventana.
cargas_180d = (
    df_jugador_fecha
    .set_index("fecha_torneo")
    .groupby("jugador_id")[columnas_volumen_fecha]
    .rolling(
        window="180D",
        closed="left"
    )
    .sum()
    .reset_index()
)


# Renombro las variables obtenidas.
cargas_180d = cargas_180d.rename(
    columns={
        "juegos_fecha": "juegos_180d",
        "sets_fecha": "sets_180d",
        "partidos_fecha": "partidos_180d",
        "torneos_fecha": "torneos_180d"
    }
)


columnas_carga_180d = [
    "juegos_180d",
    "sets_180d",
    "partidos_180d",
    "torneos_180d"
]


# Elimino las columnas si la celda se ejecuta nuevamente.
df_jugador_fecha = df_jugador_fecha.drop(
    columns=columnas_carga_180d,
    errors="ignore"
)


# Incorporo las cargas a la tabla jugador–fecha.
df_jugador_fecha = df_jugador_fecha.merge(
    cargas_180d,
    on=[
        "jugador_id",
        "fecha_torneo"
    ],
    how="left",
    validate="one_to_one"
)


# Sustituyo por cero los casos sin actividad previa.
df_jugador_fecha[columnas_carga_180d] = (
    df_jugador_fecha[columnas_carga_180d]
    .fillna(0)
    .astype("int64")
)



# Identifico el comienzo operativo de cada bloque temporal.
# 2008 y 2021 se utilizan como años buffer para construir historial.
fecha_inicio_bloque_2008_2019 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2008, 2019),
        "fecha_torneo"
    ]
    .min()
)

fecha_inicio_bloque_2021_2024 = (
    df_jugador_fecha.loc[
        df_jugador_fecha["match_year"].between(2021, 2024),
        "fecha_torneo"
    ]
    .min()
)

# Marco las observaciones de la muestra de estudio
# con 180 días completos de historial potencialmente observable.
df_jugador_fecha["historial_180d_completo"] = (
    (
        df_jugador_fecha["match_year"].between(2009, 2019)
        &
        (
            df_jugador_fecha["fecha_torneo"]
            >= fecha_inicio_bloque_2008_2019
            + pd.Timedelta(days=180)
        )
    )
    |
    (
        df_jugador_fecha["match_year"].between(2022, 2024)
        &
        (
            df_jugador_fecha["fecha_torneo"]
            >= fecha_inicio_bloque_2021_2024
            + pd.Timedelta(days=180)
        )
    )
)

# Los años buffer se usan para calcular cargas,
# pero no forman parte de la muestra de estudio.
df_jugador_fecha.loc[
    df_jugador_fecha["es_buffer_carga"].eq(1),
    "historial_180d_completo"
] = False


n_historial_180d_completo = int(
    df_jugador_fecha.loc[
        mascara_muestra_estudio,
        "historial_180d_completo"
    ].sum()
)

n_historial_180d_incompleto = (
    n_muestra_estudio - n_historial_180d_completo
)

print(
    "Observaciones jugador-fecha de la muestra de estudio:",
    n_muestra_estudio
)

print(
    "Con historial completo de 180 días:",
    n_historial_180d_completo
)

print(
    "Con historial incompleto de 180 días:",
    n_historial_180d_incompleto
)


display(
    df_jugador_fecha[
        [
            "jugador_id",
            "jugador_nombre",
            "fecha_torneo",
            "juegos_fecha",
            "juegos_180d",
            "sets_180d",
            "partidos_180d",
            "torneos_180d",
            "historial_180d_completo"
        ]
    ].head(10)
)



Observaciones jugador-fecha de la muestra de estudio: 43863
Con historial completo de 180 días: 43863
Con historial incompleto de 180 días: 0


,jugador_id,jugador_nombre,fecha_torneo,juegos_fecha,juegos_180d,sets_180d,partidos_180d,torneos_180d,historial_180d_completo
0,100644,Alexander Zverev,2013-07-15,17,0,0,0,0,True
1,100644,Alexander Zverev,2014-04-28,15,0,0,0,0,True
2,100644,Alexander Zverev,2014-07-07,26,15,2,1,1,True
3,100644,Alexander Zverev,2014-07-14,101,41,4,2,2,True
4,100644,Alexander Zverev,2014-07-21,20,142,15,7,3,True
5,100644,Alexander Zverev,2014-07-27,13,162,17,8,4,True
6,100644,Alexander Zverev,2014-10-20,26,175,19,9,5,True
7,100644,Alexander Zverev,2015-02-09,17,26,3,1,1,True
8,100644,Alexander Zverev,2015-02-16,20,43,5,2,2,True
9,100644,Alexander Zverev,2015-02-23,20,63,7,3,3,True


## Incorporación de la carga de 180 días al dataset principal

Las variables de carga de 180 días se han calculado a nivel jugador–fecha. Ahora las traslado primero a `df_jugador_torneo` y después al dataset principal `data`.

La unión se realiza utilizando `jugador_id` y `fecha_torneo`, de manera que todos los torneos disputados por un jugador en una misma fecha reciben exactamente la misma carga previa.

Las variables incorporadas serán:

* `juegos_180d`;
* `sets_180d`;
* `partidos_180d`;
* `torneos_180d`;
* `historial_180d_completo`.

El indicador `historial_180d_completo` permitirá distinguir las observaciones cuya ventana semestral se encuentra completamente cubierta por el periodo temporal disponible.


In [13]:

# ============================================================
# INCORPORACIÓN DE LA CARGA DE 180 DÍAS
# ============================================================

# Selecciono las cargas calculadas a nivel jugador–fecha.
cargas_180d_jugador_fecha = df_jugador_fecha[
    [
        "jugador_id",
        "fecha_torneo",
        "juegos_180d",
        "sets_180d",
        "partidos_180d",
        "torneos_180d",
        "historial_180d_completo"
    ]
].copy()


# Evito duplicar columnas si vuelvo a ejecutar la celda.
columnas_180d_finales = [
    "juegos_180d",
    "sets_180d",
    "partidos_180d",
    "torneos_180d",
    "historial_180d_completo"
]

df_jugador_torneo = df_jugador_torneo.drop(
    columns=columnas_180d_finales,
    errors="ignore"
)


# Incorporo la carga previa de 180 días
# a cada participación jugador–torneo.
df_jugador_torneo = df_jugador_torneo.merge(
    cargas_180d_jugador_fecha,
    on=[
        "jugador_id",
        "fecha_torneo"
    ],
    how="left",
    validate="many_to_one"
)


# Selecciono una única fila por jugador y torneo.
cargas_180d_jugador_torneo = df_jugador_torneo[
    [
        "jugador_id",
        "identificador_torneo",
        "juegos_180d",
        "sets_180d",
        "partidos_180d",
        "torneos_180d",
        "historial_180d_completo"
    ]
].copy()


# Evito duplicar columnas si vuelvo a ejecutar la celda.
data = data.drop(
    columns=columnas_180d_finales,
    errors="ignore"
)


# Incorporo las variables al dataset jugador–partido.
data = data.merge(
    cargas_180d_jugador_torneo,
    on=[
        "jugador_id",
        "identificador_torneo"
    ],
    how="left",
    validate="many_to_one"
)


print(
    "Dimensiones del dataset actualizado:",
    data.shape
)


display(
    data[
        [
            "_id_partido",
            "jugador_id",
            "jugador_nombre",
            "nombre_torneo",
            "fecha_torneo",
            "juegos_180d",
            "sets_180d",
            "partidos_180d",
            "torneos_180d",
            "historial_180d_completo"
        ]
    ].head(10)
)



Dimensiones del dataset actualizado: (94030, 93)


,_id_partido,jugador_id,jugador_nombre,nombre_torneo,fecha_torneo,juegos_180d,sets_180d,partidos_180d,torneos_180d,historial_180d_completo
0,2008-339__1,104534,Dudi Sela,Adelaide,2007-12-31,0,0,0,0,False
1,2008-339__1,103720,Lleyton Hewitt,Adelaide,2007-12-31,0,0,0,0,False
2,2008-339__2,104268,Alejandro Falla,Adelaide,2007-12-31,0,0,0,0,False
3,2008-339__2,104076,Jose Acasuso,Adelaide,2007-12-31,0,0,0,0,False
4,2008-339__3,104979,Andrey Golubev,Adelaide,2007-12-31,0,0,0,0,False
5,2008-339__3,105208,Ernests Gulbis,Adelaide,2007-12-31,0,0,0,0,False
6,2008-339__4,104542,Jo-Wilfried Tsonga,Adelaide,2007-12-31,0,0,0,0,False
7,2008-339__4,103812,Victor Hanescu,Adelaide,2007-12-31,0,0,0,0,False
8,2008-339__5,103657,Ivo Klec,Adelaide,2007-12-31,0,0,0,0,False
9,2008-339__5,103813,Jarkko Nieminen,Adelaide,2007-12-31,0,0,0,0,False


## Validación conjunta de las variables de carga competitiva

Antes de guardar el dataset final, compruebo conjuntamente las variables construidas mediante las tres definiciones temporales:

* ventana móvil de 365 días;
* acumulación desde el comienzo de la temporada;
* ventana móvil de 180 días.

La validación se centra en comprobar que:

* no existen valores ausentes en las variables de carga;
* ninguna carga presenta valores negativos;
* todos los partidos de un jugador dentro del mismo torneo reciben idénticos valores pretorneo;
* las cargas de 180 días no superan las cargas de 365 días;
* la incorporación de las nuevas variables no ha modificado el número de observaciones ni la estructura de los partidos.

La relación entre las ventanas debe cumplir:

$$
C_{i,t}^{180}(z) \leq C_{i,t}^{365}(z)
$$

porque la ventana de 180 días está completamente contenida dentro de la ventana de 365 días.

La carga de temporada no tiene que ser necesariamente inferior a la anual en todos los casos, ya que ambas variables utilizan límites temporales diferentes. La primera se reinicia al comenzar cada temporada, mientras que la segunda mantiene una ventana móvil de longitud fija.


In [14]:

# ============================================================
# VALIDACIÓN CONJUNTA DE LAS VARIABLES DE CARGA
# ============================================================

columnas_carga_365d = [
    "juegos_365d",
    "sets_365d",
    "partidos_365d",
    "torneos_365d"
]

columnas_carga_temporada = [
    "juegos_temporada",
    "sets_temporada",
    "partidos_temporada",
    "torneos_temporada"
]

columnas_carga_180d = [
    "juegos_180d",
    "sets_180d",
    "partidos_180d",
    "torneos_180d"
]

columnas_carga_totales = (
    columnas_carga_365d
    + columnas_carga_temporada
    + columnas_carga_180d
)


# Compruebo que el número de observaciones no ha cambiado.
assert len(data) == filas_iniciales, (
    "El número de observaciones ha cambiado "
    "durante la incorporación de las cargas."
)


# Compruebo que cada partido mantiene sus dos perspectivas.
assert (
    data.groupby("_id_partido")
    .size()
    .eq(2)
    .all()
), (
    "Algún partido no conserva exactamente "
    "dos observaciones."
)

assert data["_id_partido"].nunique() == partidos_iniciales

assert (
    data["es_buffer_carga"]
    + data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(1).all()

assert (
    data["es_desarrollo_modelo"]
    + data["es_validacion_temporal"]
).eq(data["es_muestra_estudio"]).all()

assert 2020 not in data["match_year"].unique()

# Compruebo que no existen valores ausentes.
assert data[columnas_carga_totales].notna().all().all(), (
    "Existen valores ausentes en las variables de carga."
)

assert data[
    [
        "historial_365d_completo",
        "historial_180d_completo"
    ]
].notna().all().all(), (
    "Existen valores ausentes en los indicadores "
    "de cobertura temporal."
)


# Compruebo que todas las cargas son no negativas.
assert data[columnas_carga_totales].ge(0).all().all(), (
    "Existen valores negativos en las variables de carga."
)


# Compruebo que la carga de 180 días nunca supera
# la carga de 365 días.
comparaciones_ventanas = [
    ("juegos_180d", "juegos_365d"),
    ("sets_180d", "sets_365d"),
    ("partidos_180d", "partidos_365d"),
    ("torneos_180d", "torneos_365d")
]

for columna_180d, columna_365d in comparaciones_ventanas:
    assert (
        data[columna_180d]
        <= data[columna_365d]
    ).all(), (
        f"{columna_180d} supera a {columna_365d} "
        "en alguna observación."
    )


# Si una observación tiene 365 días completos,
# necesariamente también debe tener 180 días completos.
assert (
    ~data["historial_365d_completo"]
    | data["historial_180d_completo"]
).all(), (
    "Existen observaciones con historial anual completo "
    "pero historial de 180 días incompleto."
)


# Compruebo que todos los partidos de un jugador
# dentro del mismo torneo reciben las mismas cargas previas.
columnas_constantes_torneo = (
    columnas_carga_totales
    + [
        "historial_365d_completo",
        "historial_180d_completo"
    ]
)

variacion_dentro_jugador_torneo = (
    data
    .groupby(
        [
            "jugador_id",
            "identificador_torneo"
        ]
    )[columnas_constantes_torneo]
    .nunique(dropna=False)
)

assert (
    variacion_dentro_jugador_torneo
    .le(1)
    .all()
    .all()
), (
    "Existen partidos del mismo jugador y torneo "
    "con cargas pretorneo diferentes."
)


print("Validación conjunta superada.")

print(
    "Observaciones con historial completo de 365 días:",
    int(data["historial_365d_completo"].sum())
)

print(
    "Observaciones con historial completo de 180 días:",
    int(data["historial_180d_completo"].sum())
)



Validación conjunta superada.
Observaciones con historial completo de 365 días: 82188
Observaciones con historial completo de 180 días: 82402


## Guardado del dataset con las variables de carga competitiva

Una vez construidas y validadas las variables de carga, guardo el dataset actualizado en un nuevo archivo CSV.

Mantengo el archivo original `df_jugador_limpio.csv` sin modificaciones. El nuevo archivo incluirá:

* las variables de carga acumulada en 365 días;
* las variables acumuladas durante la temporada;
* las variables de carga acumulada en 180 días;
* los indicadores de cobertura temporal de las ventanas móviles.

La unidad de análisis continúa siendo jugador–partido. Por tanto, cada fila conserva la información original del encuentro junto con la carga competitiva previa del jugador.

El nuevo archivo se denominará `df_jugador_cargas.csv` y constituirá la entrada de la siguiente fase, en la que se incorporarán las variables relacionadas con la participación anterior, la separación temporal y la densidad competitiva reciente.


In [15]:

# ============================================================
# GUARDADO DEL DATASET FINAL CON VARIABLES DE CARGA
# ============================================================

# Defino el nombre del archivo de salida.
ruta_salida = "df_jugador_cargas.csv"

# Calculo cuántas columnas se han añadido respecto
# al dataset limpio inicial.
columnas_anadidas = data.shape[1] - columnas_iniciales

assert len(data) == filas_iniciales
assert data["_id_partido"].nunique() == partidos_iniciales
assert columnas_anadidas == 14

# Guardo el dataset manteniendo una fila por jugador y partido.
data.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado correctamente:", ruta_salida)
print("Dimensiones finales:", data.shape)
print("Número de partidos:", data["_id_partido"].nunique())
print("Número de columnas añadidas:", columnas_anadidas)



Archivo guardado correctamente: df_jugador_cargas.csv
Dimensiones finales: (94030, 93)
Número de partidos: 47015
Número de columnas añadidas: 14


In [16]:
# Guardo las estructuras temporales utilizadas
# para construir las variables de carga.

df_jugador_torneo.to_csv(
    "df_jugador_torneo_cargas.csv",
    index=False
)

df_jugador_fecha.to_csv(
    "df_jugador_fecha_cargas.csv",
    index=False
)

print("Tablas temporales guardadas correctamente.")

Tablas temporales guardadas correctamente.


## Conclusión

Se han construido las variables de carga competitiva acumulada correspondientes a las ventanas de 365 y 180 días y a la temporada en curso, utilizando exclusivamente actividad registrada antes de la fecha de la participación actual.

El conjunto principal mantiene las 94.030 observaciones jugador–partido y los 47.015 encuentros originales, incorporando 14 variables de carga y cobertura temporal. Además, se conservan las estructuras intermedias jugador–torneo y jugador–fecha utilizadas para garantizar una construcción temporal coherente.

Los archivos `df_jugador_cargas.csv`, `df_jugador_torneo_cargas.csv` y `df_jugador_fecha_cargas.csv` constituyen la entrada de la siguiente etapa, dedicada a incorporar la carga de la participación anterior, la separación entre inicios competitivos y la densidad competitiva reciente.